In [44]:
import pandas as pd

df_pollution = pd.read_csv(r"C:/Users/PC/DATA/cleaned/pollution_clean.csv", sep=",")
df_meteo = pd.read_csv(r"C:/Users/PC/DATA/cleaned/meteo_clean.csv", sep=",")

In [45]:
df_pollution

,Date de début,code site,NO2,PM10,PM2.5,Latitude,Longitude
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0,49.119442,6.180833
1,2026-03-15 00:00:00,FR01020,28.2,13.5,11.0,49.358337,6.156942
2,2026-03-15 00:00:00,FR02022,2.1,7.9,1.9,43.675114,4.629210
3,2026-03-15 00:00:00,FR02041,2.1,3.8,1.9,43.639004,5.101097
4,2026-03-15 00:00:00,FR03006,7.3,5.6,0.9,43.276450,5.397360
...,...,...,...,...,...,...,...
65715,2026-03-30 10:00:00,FR41017,6.2,15.6,5.4,42.671333,9.434639
65716,2026-03-30 10:00:00,FR42010,12.1,2.9,1.7,48.572807,7.767833
65717,2026-03-30 10:00:00,FR50060,8.9,11.3,5.2,44.012856,1.375305
65718,2026-03-30 10:00:00,FR82010,5.2,2.7,2.1,47.510307,6.794000


### Harmoniser les colonnes

In [46]:
df_pollution.columns = df_pollution.columns.str.strip()
df_meteo.columns = df_meteo.columns.str.strip()

### Harmoniser les heures

In [47]:
print(df_pollution.columns)

Index(['Date de début', 'code site', 'NO2', 'PM10', 'PM2.5', 'Latitude',
       'Longitude'],
      dtype='object')


In [48]:
print(df_meteo.columns)

Index(['date', 'station', 'lat', 'lon', 'temperature', 'Humidité', 'wind',
       'rain'],
      dtype='object')


In [49]:
df_pollution = df_pollution.rename(columns={
    "Date de début": "date",
})
df_pollution["date"] = pd.to_datetime(df_pollution["date"], errors="coerce")
df_meteo["date"] = pd.to_datetime(df_meteo["date"], errors="coerce")

# Arrondir à l’heure
df_pollution["date"] = df_pollution["date"].dt.floor("H")
df_meteo["date"] = df_meteo["date"].dt.floor("H")

C:\Users\PC\AppData\Local\Temp\ipykernel_16656\4047859350.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pollution["date"] = df_pollution["date"].dt.floor("H")
C:\Users\PC\AppData\Local\Temp\ipykernel_16656\4047859350.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_meteo["date"] = df_meteo["date"].dt.floor("H")


### on travaille sur des copies pour éviter les effets de bord

In [50]:
df_pollution = df_pollution.copy()
df_meteo = df_meteo.copy()

### Associer station meteo la plus proche

In [59]:
df_meteo["meteo_index"] = df_meteo.index

In [60]:
from scipy.spatial import cKDTree

# Coordonnées
poll_coords = df_pollution[["Latitude", "Longitude"]].values
meteo_coords = df_meteo[["lat", "lon"]].values

tree = cKDTree(meteo_coords)

distances, indices = tree.query(poll_coords, k=1)

df_pollution["meteo_index"] = indices
df_pollution["distance_meteo"] = distances

In [52]:
df_pollution["date"] = pd.to_datetime(df_pollution["date"], errors="coerce").dt.tz_localize(None).dt.floor("H")

df_meteo["date"] = pd.to_datetime(df_meteo["date"], errors="coerce").dt.tz_localize(None).dt.floor("H")

C:\Users\PC\AppData\Local\Temp\ipykernel_16656\1220409083.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_pollution["date"] = pd.to_datetime(df_pollution["date"], errors="coerce").dt.tz_localize(None).dt.floor("H")
C:\Users\PC\AppData\Local\Temp\ipykernel_16656\1220409083.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_meteo["date"] = pd.to_datetime(df_meteo["date"], errors="coerce").dt.tz_localize(None).dt.floor("H")


In [53]:
df_pollution["date"].dtype
df_meteo["date"].dtype

dtype('<M8[ns]')

In [66]:
df_pollution = df_pollution.sort_values("date")
df_meteo = df_meteo.sort_values("date")

df_final = pd.merge_asof(
    df_pollution,
    df_meteo,
    by="meteo_index",
    on="date",
    direction="nearest"
)

In [67]:
df_final

,date,code site,NO2,PM10,PM2.5,Latitude,Longitude,meteo_index,distance_meteo,station,lat,lon,temperature,Humidité,wind,rain
0,2026-03-15 00:00:00,FR01011,7.0,4.8,1.0,49.119442,6.180833,64482,0.153349,AEROPORT METZ-NANCY-LORRAINE,48.979333,6.243167,18.2,63.0,5.7,0.0
1,2026-03-15 00:00:00,FR29439,1.7,3.7,2.9,45.481777,4.429667,10505,0.150109,ST ETIENNE-BOUTHEON,45.545667,4.293833,2.3,91.0,3.7,0.6
2,2026-03-15 00:00:00,FR24036,9.5,2.5,1.2,43.702076,7.286256,12534,0.093826,NICE,43.648833,7.209000,13.3,40.0,6.3,0.0
3,2026-03-15 00:00:00,FR24038,5.1,7.1,6.5,44.548650,6.067237,97742,0.442004,EMBRUN,44.571167,6.508667,3.6,49.0,1.6,0.0
4,2026-03-15 00:00:00,FR25036,27.7,11.7,9.3,49.442356,1.093789,39725,0.099707,ROUEN-BOOS,49.389500,1.178333,9.8,76.0,7.3,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65715,2026-03-30 10:00:00,FR08614,6.1,5.5,2.4,43.834400,4.374400,98044,0.039025,NIMES-COURBESSAC,43.856833,4.406333,6.8,38.0,5.8,0.0
65716,2026-03-30 10:00:00,FR08016,3.1,11.2,2.9,43.591500,3.886810,96334,0.079352,MONTPELLIER-AEROPORT,43.576167,3.964667,12.8,29.0,13.7,0.0
65717,2026-03-30 10:00:00,FR18043,6.2,20.1,11.3,49.259552,2.474397,17854,0.251627,ROISSY,49.015167,2.534333,8.5,92.0,5.1,0.0
65718,2026-03-30 10:00:00,FR07057,4.2,1.1,0.8,46.561974,3.340639,55214,0.399543,VICHY-CHARMEIL,46.166667,3.398667,9.8,78.0,3.9,0.0


In [68]:
df_final = df_final.drop(columns=["lat", "lon"])